# SimSat DiLoCo Round 0 Learner

Continues the DiLoCo global adapter (round 0, seeded from `simsat-gemma4-v10-adapter`) on the review-refreshed SimSat ChatML dataset (`benhaslam/simsat-gemma4-v1` v2 — 713 rows, REFINE_BOOST=1.5).

**Inputs (all attached):**
1. Model: `google/gemma-4` Transformers → `gemma-4-e2b-it/1`
2. Dataset: `benhaslam/simsat-gemma4-v1` (training JSONL)
3. Dataset: `benhaslam/diloco-lab-src` (DiLoCo source — auto-extracted by Kaggle)
4. Dataset: `benhaslam/diloco-global-round-000000` (round-0 global adapter — auto-extracted)

**Outputs:**
- `/kaggle/working/diloco_continued_adapter/` — continued LoRA + tokenizer + summary
- `/kaggle/working/diloco_outbox/<experiment>/round-000000/<learner>/` — fragment deltas

**v3 fix:** stage a writable copy of the seed adapter and flip
`inference_mode: false` in `adapter_config.json` before the runner attaches it.
Round 0 v2 produced an adapter byte-identical to the seed because PEFT loaded
the LoRA in inference mode and `save_pretrained` serialized the load-time
weights regardless of optimizer steps. Also adds a post-train sha256 check
that fails loudly if the saved adapter still matches the seed.

In [ ]:
import hashlib, json, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')

def _resolve_source(prefer_dir: str, zip_name: str, marker: str) -> Path:
    """Kaggle auto-extracts zips on dataset upload; some setups still ship raw .zip.
    Prefer the extracted-dir form (find `marker` anywhere under /kaggle/input);
    fall back to extracting `zip_name` to /kaggle/working if no extracted copy.
    Returns the directory containing `marker`."""
    direct = list(INPUT.rglob(marker))
    if direct:
        return direct[0].parent
    zips = list(INPUT.rglob(zip_name))
    if not zips:
        raise RuntimeError(f'Neither extracted {marker} nor {zip_name} found under /kaggle/input.')
    target = WORK / prefer_dir
    if not target.exists():
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(target)
    return next(target.rglob(marker)).parent

def _sha256(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

# --- Source bundle ---
SRC = _resolve_source('diloco_lab_src', 'diloco_lab_source.zip', 'continue_gemma4_adapter.py').parent
print(f'SRC: {SRC}')

# --- Global adapter: stage a writable copy + flip inference_mode ---
READ_ONLY_ADAPTER = _resolve_source('global_adapter', 'global_round_000000.zip', 'adapter_config.json')
ADAPTER = WORK / 'global_adapter_writable'
if ADAPTER.exists():
    shutil.rmtree(ADAPTER)
shutil.copytree(READ_ONLY_ADAPTER, ADAPTER)

config_path = ADAPTER / 'adapter_config.json'
config = json.loads(config_path.read_text())
previous = config.get('inference_mode')
config['inference_mode'] = False
config_path.write_text(json.dumps(config, indent=2))
print(f'Adapter staged: {ADAPTER}  (inference_mode: {previous!r} -> False)')

SEED_HASH = _sha256(ADAPTER / 'adapter_model.safetensors')
print(f'Seed adapter sha256: {SEED_HASH}')

# --- Run the DiLoCo continuation ---
subprocess.check_call([
    sys.executable,
    str(SRC / 'kaggle' / 'continue_gemma4_adapter.py'),
    '--base-adapter', str(ADAPTER),
    '--project', 'simsat',
    '--dataset-id', 'simsat-gemma4-v3-reviewed',
    '--learner-id', 'kaggle-t4-simsat-round0-a',
    '--round-id', '0',
    '--max-steps', '120',
    '--lr', '5e-5',
])

# --- Post-training sanity: continued adapter MUST differ from seed ---
out = WORK / 'diloco_continued_adapter' / 'adapter_model.safetensors'
OUT_HASH = _sha256(out)
print(f'Continued sha256:    {OUT_HASH}')
if OUT_HASH == SEED_HASH:
    raise RuntimeError(
        'Continued adapter is bit-identical to seed. Training did not persist. '
        'Inspect the runner save path (continue_gemma4_adapter.py:_train -> model.save_pretrained).'
    )
print('OK: continued adapter differs from seed.')